In [ ]:

import requests
import time
import statistics

def naive_search(text, pattern):
    return [i for i in range(len(text) - len(pattern) + 1)
            if text[i:i + len(pattern)] == pattern]

def kmp_search(text, pattern):
    m = len(pattern)
    lps = [0] * m
    j = 0
    for i in range(1, m):
        while j and pattern[i] != pattern[j]:
            j = lps[j - 1]
        if pattern[i] == pattern[j]:
            j += 1
            lps[i] = j

    positions = []
    i = j = 0
    while i < len(text):
        if text[i] == pattern[j]:
            i += 1
            j += 1
        if j == m:
            positions.append(i - j)
            j = lps[j - 1]
        elif i < len(text) and text[i] != pattern[j]:
            if j:
                j = lps[j - 1]
            else:
                i += 1
    return positions

def rabin_karp(text, pattern):
    n, m = len(text), len(pattern)
    base, prime = 256, 101
    ph = th = 0
    power = pow(base, m - 1, prime)
    positions = []

    for i in range(m):
        ph = (base * ph + ord(pattern[i])) % prime
        th = (base * th + ord(text[i])) % prime

    for i in range(n - m + 1):
        if ph == th and text[i:i + m] == pattern:
            positions.append(i)
        if i < n - m:
            th = (base * (th - ord(text[i]) * power)
                  + ord(text[i + m])) % prime
    return positions

algorithms = {
    "Naive": naive_search,
    "KMP": kmp_search,
    "Rabin-Karp": rabin_karp
}

def wikipedia_text(title):
    url = "https://en.wikipedia.org/w/rest.php/v1/page/" + requests.utils.quote(title, safe="")
    response = requests.get(
        url,
        headers={"User-Agent": "StringMatchingStudentProject/1.0"},
        timeout=20
    )
    response.raise_for_status()
    data = response.json()
    return data["source"]

def train(text):
    print(" TRAINING")
    samples = [
        text[:500],
        text[500:1000],
        text[1000:1500]
    ]
    samples = [s for s in samples if s]

    for i, sample in enumerate(samples, 1):
        pattern = sample[:min(8, len(sample))]
        print(f"Training sample {i}: {len(sample)} characters")
        for name, func in algorithms.items():
            start = time.perf_counter()
            func(sample, pattern)
            elapsed = time.perf_counter() - start
            print(f"{name}: {elapsed:.8f} seconds")

    print("Training benchmark completed.")

def select_algorithm(text, pattern):
    if len(pattern) <= 3:
        return "Naive"
    if len(set(text)) <= 4 or len(text) >= 10000:
        return "KMP"
    return "Rabin-Karp"

def test(text):
    print("TESTING")
    pattern = input("Enter pattern to search: ")

    if not pattern:
        print("Pattern cannot be empty.")
        return

    if len(pattern) > len(text):
        print("Pattern is longer than the text.")
        return

    selected = select_algorithm(text, pattern)
    print("Selected algorithm:", selected)

    expected = None
    results = {}

    for name, func in algorithms.items():
        start = time.perf_counter()
        positions = func(text, pattern)
        elapsed = time.perf_counter() - start
        results[name] = positions

        if expected is None:
            expected = positions

        print(f"\n{name}")
        print("Matching positions:", positions[:20])
        print("Total matches:", len(positions))
        print("Execution time:", f"{elapsed:.8f} seconds")
        print("Correct:", positions == expected)

    print("\nPredicted algorithm:", selected)
    print("Text length:", len(text))
    print("Pattern length:", len(pattern))
    print("Wikipedia dataset: downloaded article text")

title = input("Enter Wikipedia article title (e.g. Python): ")

try:
    text = wikipedia_text(title)
    print("\nWikipedia article downloaded:", title)
    print("Dataset size:", len(text), "characters")

    train(text)

    while True:
        test(text)
        if input("\nTest another pattern? (y/n): ").lower() != "y":
            break

except (requests.RequestException, KeyError, ValueError) as e:
    print("Unable to download Wikipedia article:", e)


Enter Wikipedia article title (e.g. Python): Artificial intelligence

Wikipedia article downloaded: Artificial intelligence
Dataset size: 283885 characters
 TRAINING
Training sample 1: 500 characters
Naive: 0.00007400 seconds
KMP: 0.00011316 seconds
Rabin-Karp: 0.00013237 seconds
Training sample 2: 500 characters
Naive: 0.00007043 seconds
KMP: 0.00010375 seconds
Rabin-Karp: 0.00013284 seconds
Training sample 3: 500 characters
Naive: 0.00006223 seconds
KMP: 0.00013005 seconds
Rabin-Karp: 0.00012651 seconds
Training benchmark completed.
TESTING
Enter pattern to search: intelligence
Selected algorithm: KMP

Naive
Matching positions: [107, 204, 388, 522, 871, 1058, 2360, 2596, 2796, 2903, 3102, 3284, 4092, 4214, 4305, 4404, 4495, 7138, 9397, 9447]
Total matches: 248
Execution time: 0.05725378 seconds
Correct: True

KMP
Matching positions: [107, 204, 388, 522, 871, 1058, 2360, 2596, 2796, 2903, 3102, 3284, 4092, 4214, 4305, 4404, 4495, 7138, 9397, 9447]
Total matches: 248
Execution time: 0.

In [ ]:
import time
import statistics
import numpy as np

def evaluate_model(text, pattern, trials=10):
    algorithms = {
        "Naive": naive_search,
        "KMP": kmp_search,
        "Rabin-Karp": rabin_karp
    }

    reference = naive_search(text, pattern)
    timings = {}
    results = {}

    print(" EVALUATION ")

    for name, func in algorithms.items():
        runs = []

        for _ in range(trials):
            start = time.perf_counter()
            output = func(text, pattern)
            runs.append(time.perf_counter() - start)

        timings[name] = runs
        results[name] = output

        print(f"\nAlgorithm: {name}")
        print("Correctness:", output == reference)
        print("Total Matches:", len(output))
        print("Mean Time:", f"{statistics.mean(runs):.8f} seconds")
        print("Median Time:", f"{statistics.median(runs):.8f} seconds")
        print("Std Deviation:", f"{statistics.pstdev(runs):.8f} seconds")
        print("Minimum Time:", f"{min(runs):.8f} seconds")
        print("Maximum Time:", f"{max(runs):.8f} seconds")

    fastest = min(
        timings,
        key=lambda name: statistics.median(timings[name])
    )

    selected = select_algorithm(text, pattern)
    baseline = statistics.median(timings["Naive"])
    selected_time = statistics.median(timings[selected])
    speedup = baseline / selected_time if selected_time > 0 else 0

    print(" PERFORMANCE SUMMARY")
    print("Dataset Size:", len(text), "characters")
    print("Pattern Length:", len(pattern))
    print("Alphabet Size:", len(set(text)))
    print("Total Occurrences:", len(reference))
    print("Fastest Algorithm:", fastest)
    print("Selected Algorithm:", selected)
    print("Selector Optimal:", selected == fastest)
    print("Speedup vs Naive:", f"{speedup:.3f}x")
    print("All Algorithms Correct:",
          all(result == reference for result in results.values()))

    print(" RELATIVE RUNTIME")
    for name in algorithms:
        median = statistics.median(timings[name])
        ratio = median / baseline if baseline > 0 else 0
        print(f"{name}: {ratio:.3f}x Naive runtime")

pattern = "intelligence"
evaluate_model(text, pattern)

 EVALUATION 

Algorithm: Naive
Correctness: True
Total Matches: 248
Mean Time: 0.04534721 seconds
Median Time: 0.04503857 seconds
Std Deviation: 0.00121992 seconds
Minimum Time: 0.04409235 seconds
Maximum Time: 0.04822911 seconds

Algorithm: KMP
Correctness: True
Total Matches: 248
Mean Time: 0.06219923 seconds
Median Time: 0.06021139 seconds
Std Deviation: 0.00498396 seconds
Minimum Time: 0.05964244 seconds
Maximum Time: 0.07691094 seconds

Algorithm: Rabin-Karp
Correctness: True
Total Matches: 248
Mean Time: 0.07650874 seconds
Median Time: 0.07616506 seconds
Std Deviation: 0.00109331 seconds
Minimum Time: 0.07544058 seconds
Maximum Time: 0.07870894 seconds
 PERFORMANCE SUMMARY
Dataset Size: 283885 characters
Pattern Length: 12
Alphabet Size: 137
Total Occurrences: 248
Fastest Algorithm: Naive
Selected Algorithm: KMP
Selector Optimal: False
Speedup vs Naive: 0.748x
All Algorithms Correct: True
 RELATIVE RUNTIME
Naive: 1.000x Naive runtime
KMP: 1.337x Naive runtime
Rabin-Karp: 1.691x N